# AML Rules and ML Triage MVP: End-to-End Walkthrough

This notebook demonstrates the end-to-end workflow of the Anti-Money Laundering (AML) Rules and Machine Learning (ML) Triage system. It is designed to prioritize suspicious activity alerts generated by deterministic rule coverage using ML model rankings.

### Architecture Overview
1. **Ingestion & Ingestion Data Quality Checks**: Standardizing raw data and checking data health.
2. **Temporal Split Management**: Partitioning transaction records sequentially to prevent label leakage.
3. **AML Rule Engine (R1 - R6)**: Generating high-coverage, auditable alerts based on historical behavioral patterns (Amount, Novelty, Velocity, Pass-through, Fan-in, Fan-out).
4. **Threshold Tuning**: Optimizing continuous rule thresholds subject to validation recall floors.
5. **Alert Feature Engineering**: Building model-ready alert-level features (historical account history, rule hits, graph-lite topology).
6. **ML Triage Training & Scoring**: Fitting ML classification models (logistic regression and gradient boosting classifiers) on alert features to rank risk.
7. **Champion vs Challenger Comparison**: Auditing performance delta against a graph-augmented model.
8. **Priority Banding & Capacity Simulation**: Mapping scores to operationally clear P1-P4 queues and evaluating workforce throughput.
9. **SHAP Explainability**: Translating ML coefficients and SHAP values into investigator-friendly reason codes.

## 1. Setup Environment and Configurations

First, we adjust python paths and load the project YAML configurations which coordinate data paths, splits, rules, and model hyperparameters.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import json

# Adjust path to find the aml_mvp modules and config files
if Path.cwd().name == "notebooks":
    os.chdir("..")
if "src" not in sys.path:
    sys.path.append("src")

from aml_mvp.config import load_config

# Load configuration files
data_config = load_config("config/data_config.yaml")
model_config = load_config("config/model_config.yaml")
rule_config = load_config("config/rule_config.yaml")
extended_config = load_config("config/extended_config.yaml")

print("Successfully loaded configurations:")
print(f"  Dataset Name:        {data_config['data']['dataset_name']}")
print(f"  Primary ML Target:   {model_config['model']['target']}")
print(f"  Enabled Rules Count: {len(rule_config['rules']['enabled'])}")

## 2. Ingestion, Splits, and Data Profiling

We inspect the dataset's splits and boundaries. Temporal splitting is essential in AML to ensure that models are evaluated on future data relative to their training partition.

In [ ]:
# Load and display the split manifest
with open("outputs/metrics/split_manifest.json", "r") as f:
    split_manifest = json.load(f)

print(json.dumps(split_manifest, indent=2))

## 3. AML Rule Engine Execution (Mini-Demo & Precomputed Performance)

The rule engine processes transactions to generate alerts. Below, we demonstrate the API execution on a toy transaction set to show how rules trigger, and then we inspect the overall rule coverage performance and overlap on the full dataset.

In [ ]:
from aml_mvp.rules.rule_engine import run_rule_engine

# Define custom toy transactions representing different typologies:
# - txn 1: standard cross-bank transfer
# - txn 2: R1 AMOUNT trigger (large transaction relative to ACH/USD)
# - txn 3: R2 NOVELTY trigger (new sender-receiver pair)
toy_tx = pd.DataFrame({
    "transaction_id": [1, 2, 3, 4],
    "timestamp": pd.to_datetime([
        "2026-05-01 00:00:00",
        "2026-05-01 01:00:00",
        "2026-05-01 02:00:00",
        "2026-05-01 03:00:00"
    ]),
    "from_bank": ["BankA", "BankB", "BankA", "BankC"],
    "to_bank": ["BankB", "BankB", "BankC", "BankA"],
    "sender_account_id": ["S100", "S200", "S100", "S300"],
    "receiver_account_id": ["R200", "R300", "R300", "R100"],
    "amount": [500.0, 100000.0, 1200.0, 450.0],
    "amount_paid": [500.0, 100000.0, 1200.0, 450.0],
    "amount_received": [500.0, 100000.0, 1200.0, 450.0],
    "payment_currency": ["USD"] * 4,
    "receiving_currency": ["USD"] * 4,
    "payment_format": ["ACH"] * 4,
    "currency_pair": ["USD -> USD"] * 4,
    "cross_bank_flag": [1, 0, 1, 1],
    "log_amount": [6.21, 11.51, 7.09, 6.11],
    "is_laundering": [0, 1, 0, 0],
    "split": ["train", "train", "validation", "test"]
})

# Run the rule engine
rule_hits_toy, alerts_toy = run_rule_engine(toy_tx, rule_config)

print("=== RULE HITS TOY OUTPUT ===")
print(rule_hits_toy[["transaction_id", "rule_id", "severity", "rationale"]])

print("\n=== CONSOLIDATED ALERTS TOY OUTPUT ===")
print(alerts_toy[["alert_id", "transaction_id", "triggered_rules", "rule_count", "max_rule_severity"]])

In [ ]:
# Load rule performance metrics on the full validation/test set
rule_perf = pd.read_csv("outputs/metrics/rule_performance.csv")
print("=== RULE PERFORMANCE (FULL DATASET) ===")
print(rule_perf)

# Load rule overlap matrix to study coverage redundance
overlap_matrix = pd.read_csv("outputs/metrics/rule_overlap_matrix.csv")
print("\n=== RULE OVERLAP MATRIX ===")
print(overlap_matrix)

## 4. Alert Feature Matrix

For ML triage, features are engineered at the alert level (rather than individual transaction level). We check the schema and description of features in the feature dictionary.

In [ ]:
# Load feature dictionary
feat_dict = pd.read_csv("outputs/metrics/feature_dictionary.csv")
print("=== FEATURE DICTIONARY ===")
print(feat_dict[["feature_name", "feature_group", "definition"]])

## 5. ML Triage Model Performance and Ranking Evaluation

We inspect the evaluation metrics of the model. We plot the Precision@K and Recall@K curves to visualize the model's triage capability compared to baselines.

In [ ]:
import matplotlib.pyplot as plt

# Load model top-K metrics
top_k_df = pd.read_csv("outputs/metrics/top_k_metrics.csv")
print("=== TOP-K PERFORMANCE COMPARISON ===")
print(top_k_df)

# Plot Precision@K and Recall@K curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision@K
for name, group in top_k_df.groupby("ranking"):
    axes[0].plot(group["k"].astype(str), group["precision_at_k"], marker='o', label=name, linewidth=2)
axes[0].set_title("Precision @ K Comparison", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Capacity (K)", fontsize=10)
axes[0].set_ylabel("Precision", fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# Recall@K
for name, group in top_k_df.groupby("ranking"):
    axes[1].plot(group["k"].astype(str), group["recall_at_k"], marker='s', label=name, linewidth=2)
axes[1].set_title("Recall @ K Comparison", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Capacity (K)", fontsize=10)
axes[1].set_ylabel("Recall", fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Champion vs Challenger Model Auditing (Extended Graph Features)

Here we compare the performance of the baseline MVP model against the challenger model which leverages graph topological features (PageRank, Cycle involvement, gather-scatter metrics).

In [ ]:
# Load model comparison metrics
comp_df = pd.read_csv("outputs/extended/model_comparison.csv")
print("=== MVP VS EXTENDED MODEL METRICS ===")
print(comp_df[["metric", "k", "mvp_value", "extended_value", "delta", "winner"]])

# Load model selection decision
with open("outputs/extended/model_selection.json", "r") as f:
    selection = json.load(f)

print("\n=== DECISION MAKING SUMMARY ===")
print(f"Selected Model: {selection['selected_model'].upper()}")
print(f"Decision Rule:  {selection['decision']}")
print(f"Explanation:    {selection['reason']}")

## 7. Priority Banding and Workforce Capacity Simulation

Alerts are routed to four priority bands (P1-P4). High-severity event rules (like R4 Rapid Pass-Through) override model rankings to guarantee they are never buried.

In [ ]:
# Load priority band summary
band_summary = pd.read_csv("outputs/metrics/band_summary.csv")
print("=== PRIORITY BAND DISTRIBUTION ===")
print(band_summary)

# Load capacity simulation
cap_sim = pd.read_csv("outputs/metrics/capacity_simulation.csv")

# Plot precision at different capacity limits
fig, ax = plt.subplots(figsize=(9, 5))
for name, group in cap_sim.groupby("ranking"):
    ax.plot(group["k"].astype(str), group["precision_at_k"], marker='o', label=name, linewidth=2)
ax.set_title("Workforce Precision vs Review Capacity", fontsize=12, fontweight='bold')
ax.set_xlabel("Number of Alerts Reviewed (K)", fontsize=10)
ax.set_ylabel("Precision (True Positives Rate)", fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()
plt.show()

## 8. SHAP Explainability & Reason Codes

To support compliance officers in their audits, we decompose model scores into feature-level contributions. We plot the global SHAP feature importances and inspect the generated alert reason codes.

In [ ]:
# Load global SHAP feature importances
shap_importance = pd.read_csv("outputs/extended/shap_feature_importance.csv")

# Plot top 10 features by SHAP importance
plt.figure(figsize=(10, 5))
top_10 = shap_importance.sort_values("mean_abs_shap", ascending=True).tail(10)
plt.barh(top_10["feature_name"], top_10["mean_abs_shap"], color='#2c3e50')
plt.title("Top 10 Feature Contributions (SHAP Importances)", fontsize=12, fontweight='bold')
plt.xlabel("Mean |SHAP Value| (Impact on Model Score)", fontsize=10)
plt.grid(True, axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Load reason codes sample
reasons = pd.read_csv("outputs/extended/reason_codes.csv")
print("=== SAMPLE REASON CODES FOR AUDITING ===")
print(reasons.head(9))